In [0]:
import dlt
from pyspark.sql.functions import *
from pyspark.sql.window import Window

def read_silver(table):
    return spark.read.table(f"retail_store_sales.silver.{table}")


@dlt.table(name="dim_customer",table_properties={"pipelines.autoOptimize.managed": "true"}, comment="Customer dimension (SCD2 current)")
def dim_customer():
    return (
        read_silver("customers_scd")
        .filter("__END_AT IS NULL")
        .select(
            "customer_sk",
            "customer_id",
            "customer_name",
            col("city").alias("customer_city"),
            "loyalty_tier",
            "email_masked",
            "phone_masked"
        )
    )

@dlt.table(name="dim_product",table_properties={"pipelines.autoOptimize.managed": "true"}, comment="Product dimension (SCD2 current)")
def dim_product():
    return (
        read_silver("products_scd")
        .filter("__END_AT IS NULL")
        .select(
            "product_sk",
            "product_id",
            "product_name",
            "category",
            col("price").alias("list_price")
        )
    )

@dlt.table(name="dim_store", comment="Store dimension (SCD2 current)")
def dim_store():
    return (
        read_silver("stores_scd")
        .filter("__END_AT IS NULL")
        .select(
            "store_sk",
            "store_id",
            "store_name",
            col("city").alias("store_city")
        )
    )

@dlt.table(name="fact_sales", table_properties={"pipelines.autoOptimize.managed": "true"}, comment="Sales fact table with surrogate keys")
def fact_sales():

    sales = read_silver("sales_silver")
    cust = read_silver("customers_scd").filter("__END_AT IS NULL")
    prod = read_silver("products_scd").filter("__END_AT IS NULL")
    store = read_silver("stores_scd").filter("__END_AT IS NULL")

    return (
        sales
        .join(cust.select("customer_id", "customer_sk"), "customer_id", "left")
        .join(prod.select("product_id", "product_sk", "product_name"), "product_id", "left")
        .join(store.select("store_id", "store_sk"), "store_id", "left")
        .select(
            "order_id",
            "order_date",
            "customer_sk",
            "product_sk",
            "product_name",
            "category",
            "store_sk",
            "quantity",
            "price_per_unit",
            "discount_applied",
            "total_amount",
            "payment_method",
            current_timestamp().alias("ingestion_ts")
        )
    )

@dlt.table(name="customer_360", table_properties={"pipelines.autoOptimize.managed": "true"}, comment="Unified customer 360 view")
def customer_360():

    fact = dlt.read("fact_sales")
    dim = dlt.read("dim_customer")

    agg = (
        fact.groupBy("customer_sk")
        .agg(
            round(sum("total_amount"), 2).alias("total_spend"),
            count("order_id").alias("total_orders"),
            round(avg("total_amount"), 2).alias("avg_order_value"),
            max("order_date").alias("last_order_date"),
            min("order_date").alias("first_order_date"),
            countDistinct("store_sk").alias("stores_visited"),
            sum("quantity").alias("items_purchased")
        )
        .withColumn(
            "customer_segment",
            when(col("total_spend") >= 2000, "Platinum")
            .when(col("total_spend") >= 1000, "Gold")
            .when(col("total_spend") >= 500, "Silver")
            .otherwise("Bronze")
        )
    )

    return (
        agg.join(dim, "customer_sk", "left")
    )

@dlt.table(name="daily_store_sales", table_properties={"pipelines.autoOptimize.managed": "true"}, comment="Daily store performance")
def daily_store_sales():

    fact = dlt.read("fact_sales")
    store = dlt.read("dim_store")

    return (
        fact
        .groupBy("store_sk", "order_date")
        .agg(
            round(sum("total_amount"), 2).alias("daily_revenue"),
            count("order_id").alias("total_orders"),
            sum("quantity").alias("units_sold")
        )
        .join(store, "store_sk", "left")
    )

@dlt.table(name="monthly_revenue_trend", table_properties={"pipelines.autoOptimize.managed": "true"})
def monthly_revenue_trend():

    fact = dlt.read("fact_sales")

    return (
        fact
        .withColumn("year", year("order_date"))
        .withColumn("month", month("order_date"))
        .groupBy("year", "month")
        .agg(
            round(sum("total_amount"), 2).alias("revenue"),
            count("order_id").alias("orders"),
            round(avg("total_amount"), 2).alias("avg_order_value")
        )
    )

@dlt.table(name="top_products", table_properties={"pipelines.autoOptimize.managed": "true"})
def top_products():

    fact = dlt.read("fact_sales")
    prod = dlt.read("dim_product")

    return (
        fact.groupBy("product_sk")
        .agg(
            sum("quantity").alias("units_sold"),
            round(sum("total_amount"), 2).alias("revenue")
        )
        .join(prod, "product_sk")
        .orderBy(col("revenue").desc())
    )

@dlt.table(name="payment_analysis", table_properties={"pipelines.autoOptimize.managed": "true"})
def payment_analysis():

    fact = dlt.read("fact_sales")

    total = fact.agg(sum("total_amount").alias("total"))

    return (
        fact.groupBy("payment_method")
        .agg(sum("total_amount").alias("revenue"))
        .crossJoin(total)
        .withColumn("share_pct", col("revenue") / col("total") * 100)
    )

@dlt.table(name="inventory_risk")
def inventory_risk():

    inv = read_silver("inventory_events_silver")
    prod = dlt.read("dim_product")

    return (
        inv.groupBy("product_id")
        .agg(min("stock_level").alias("min_stock"))
        .join(prod, "product_id")
        .withColumn(
            "risk",
            when(col("min_stock") == 0, "CRITICAL")
            .when(col("min_stock") < 20, "LOW")
            .otherwise("SAFE")
        )
    )

@dlt.table(
    name="store_inventory_alerts",
    comment="Real-time stock alerts per store and product",
    table_properties={"pipelines.autoOptimize.managed": "true","quality": "gold"}, 
)
def store_inventory_alerts():

    inventory = read_silver("inventory_events_silver")
    products = dlt.read("dim_product")
    stores = dlt.read("dim_store")

    # Get latest stock per product-store
    latest_stock = (
        inventory
        .withColumn(
            "rn",
            row_number().over(
                Window.partitionBy("product_id", "store_id")
                .orderBy(col("event_time").desc())
            )
        )
        .filter("rn = 1")
        .drop("rn")
    )

    return (
        latest_stock
        .join(products.select("product_id", "product_name", "category", "list_price"), "product_id", "left")
        .join(stores.select("store_id", "store_name", "store_city"), "store_id", "left")

        # Alert Logic
        .withColumn(
            "alert_level",
            when(col("stock_level") == 0, "OUT OF STOCK")
            .when(col("stock_level") <= 10, "CRITICAL")
            .when(col("stock_level") <= 30, "LOW")
            .when(col("stock_level") <= 75, "MODERATE")
            .otherwise("HEALTHY")
        )

        # Priority (important for dashboards)
        .withColumn(
            "alert_priority",
            when(col("stock_level") == 0, 1)
            .when(col("stock_level") <= 10, 2)
            .when(col("stock_level") <= 30, 3)
            .when(col("stock_level") <= 75, 4)
            .otherwise(5)
        )

        # Business impact
        .withColumn(
            "estimated_stock_value",
            round(col("stock_level") * col("list_price"), 2)
        )

        .withColumn("ingestion_ts", current_timestamp())
    )

@dlt.table(name="discount_impact_analysis", table_properties={"pipelines.autoOptimize.managed": "true"}, comment="Impact of discounts on revenue")
def discount_impact_analysis():

    fact = dlt.read("fact_sales")

    return (
        fact
        .withColumn(
            "before_discount_revenue",
            col("quantity") * col("price_per_unit")
        )
        .groupBy("order_date")
        .agg(
            round(sum("before_discount_revenue"), 2).alias("before_discount"),
            round(sum("total_amount"), 2).alias("after_discount"),
            round(
                sum("before_discount_revenue") - sum("total_amount"), 2
            ).alias("discount_loss"),
            round(
                (sum("before_discount_revenue") - sum("total_amount")) 
                / sum("before_discount_revenue") * 100, 2
            ).alias("discount_percentage")
        )
    )